In [1]:
import pandas as pd
import numpy as np
import os

os.chdir('/Users/liujingxin/Documents/Quant_Project')

# 读取上一步保存的数据
# parse_dates 告诉 pandas 把 trade_date 这列直接读成日期格式，不用再转换
df = pd.read_csv('data/raw_monthly_data.csv', parse_dates=['trade_date'])

print(f'原始数据：{len(df):,} 行，{df["ts_code"].nunique():,} 只股票')
df.head()

原始数据：349,215 行，5,675 只股票


,ts_code,trade_date,close,pe_ttm,pb,total_mv,circ_mv,industry
0,000001.SZ,2020-01-23,15.54,10.6958,1.1045,3.015680e+07,3.015654e+07,银行
1,000001.SZ,2020-02-28,14.50,9.9800,1.0306,2.813858e+07,2.813834e+07,银行
2,000001.SZ,2020-03-31,12.80,8.8099,0.9098,2.483958e+07,2.483936e+07,银行
3,000001.SZ,2020-04-30,13.93,9.2270,0.9572,2.703244e+07,2.703221e+07,银行
4,000001.SZ,2020-05-29,13.00,8.6110,0.7687,2.522769e+07,2.522748e+07,银行


In [2]:
# 记录每一步过滤掉了多少行，方便你理解数据变化
print(f'清洗前：{len(df):,} 行')

# 第一步：删除 pe_ttm 或 pb 缺失的行
# 这两个是我们构建因子的核心数据，缺了就没法用
# dropna() 是"删除含有缺失值的行"的函数
# subset 指定只看这两列
df = df.dropna(subset=['pe_ttm', 'pb'])
print(f'删除 PE/PB 缺失后：{len(df):,} 行')

# 第二步：删除 pe_ttm 或 pb 为负数或零的行
# PE 负数代表亏损公司，EP 因子对这类公司没有意义
# PB 负数代表资不抵债，同样排除
# 用布尔索引筛选：只保留 pe_ttm > 0 且 pb > 0 的行
df = df[(df['pe_ttm'] > 0) & (df['pb'] > 0)]
print(f'删除 PE/PB 负值后：{len(df):,} 行')

# 第三步：删除行业缺失的行
# 我们要做行业中性化，没有行业分类的股票没法处理
df = df.dropna(subset=['industry'])
print(f'删除行业缺失后：{len(df):,} 行')

# 第四步：过滤新股
# 新上市不足 6 个月的股票数据不稳定，排除掉
# 思路：用上市日期和当前交易日期对比，如果差距小于 180 天就过滤
# 先读取股票基本信息（里面有上市日期）
stock_info = pd.read_csv('data/stock_industry.csv', parse_dates=['list_date'])

# 只保留股票代码和上市日期两列
stock_info = stock_info[['ts_code', 'list_date']]

# 把上市日期合并进 df
df = df.merge(stock_info, on='ts_code', how='left')

# 计算上市天数：交易日期 - 上市日期，取天数
df['days_listed'] = (df['trade_date'] - df['list_date']).dt.days

# 只保留上市超过 180 天的股票
df = df[df['days_listed'] >= 180]
print(f'过滤新股（上市<180天）后：{len(df):,} 行')

# 删除 list_date 和 days_listed 这两列，后面用不到了
df = df.drop(columns=['list_date', 'days_listed'])

print(f'\n清洗完成，剩余 {len(df):,} 行，{df["ts_code"].nunique():,} 只股票')

清洗前：349,215 行
删除 PE/PB 缺失后：275,097 行
删除 PE/PB 负值后：275,097 行
删除行业缺失后：271,971 行
过滤新股（上市<180天）后：261,075 行

清洗完成，剩余 261,075 行，5,338 只股票


In [3]:
# Winsorize 是什么？
# 我们在 Cell 9 看到 pe_ttm 最大值是 539273，pb 最大值是 9234
# 这些极端值会严重干扰因子排序
# Winsorize 的做法：把超过 99% 分位数的值，强制设为 99% 分位数的值
# 把低于 1% 分位数的值，强制设为 1% 分位数的值
# 相当于"把两端的极端值往中间拉"

# 我们对每个月截面内做 Winsorize（不是对全部数据）
# 原因：不同时期市场估值水平不同，应该在同一时间点内比较

def winsorize_monthly(group, col, lower=0.01, upper=0.99):
    """
    对某一列做月度 Winsorize
    group: 某个月的数据（DataFrame）
    col: 要处理的列名
    lower: 下界分位数，默认 1%
    upper: 上界分位数，默认 99%
    """
    # 计算这个月这列数据的 1% 和 99% 分位数
    lo = group[col].quantile(lower)
    hi = group[col].quantile(upper)
    
    # clip() 函数：把低于 lo 的值设为 lo，高于 hi 的值设为 hi
    group[col] = group[col].clip(lower=lo, upper=hi)
    return group

# 对 pe_ttm 和 pb 分别按月做 Winsorize
# groupby('trade_date') 是按月分组（我们的数据里每个 trade_date 是月末）
# apply() 对每个分组应用我们定义的函数
print('正在对 pe_ttm 做 Winsorize...')
df = df.groupby('trade_date', group_keys=False).apply(
    lambda g: winsorize_monthly(g, 'pe_ttm')
)

print('正在对 pb 做 Winsorize...')
df = df.groupby('trade_date', group_keys=False).apply(
    lambda g: winsorize_monthly(g, 'pb')
)

print('Winsorize 完成 ✓')
print(f'\npe_ttm 处理后范围：{df["pe_ttm"].min():.2f} 到 {df["pe_ttm"].max():.2f}')
print(f'pb 处理后范围：{df["pb"].min():.2f} 到 {df["pb"].max():.2f}')

正在对 pe_ttm 做 Winsorize...


/var/folders/vd/j4l_2tgj3mqb004b4l__22sh0000gn/T/ipykernel_62744/1560075091.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('trade_date', group_keys=False).apply(


正在对 pb 做 Winsorize...
Winsorize 完成 ✓

pe_ttm 处理后范围：4.03 到 1627.09
pb 处理后范围：0.48 到 26.65


/var/folders/vd/j4l_2tgj3mqb004b4l__22sh0000gn/T/ipykernel_62744/1560075091.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('trade_date', group_keys=False).apply(


In [4]:
# EP 因子 = 1 / PE_ttm
# 含义：每一块钱市值能对应多少盈利
# EP 越高 → 股票越便宜（盈利相对于股价高）
df['EP'] = 1 / df['pe_ttm']

# BP 因子 = 1 / PB
# 含义：每一块钱市值能对应多少账面净资产
# BP 越高 → 股票越便宜（账面价值相对于股价高）
df['BP'] = 1 / df['pb']

print('原始因子构建完成 ✓')
print(f'\nEP 统计：')
print(df['EP'].describe())
print(f'\nBP 统计：')
print(df['BP'].describe())

原始因子构建完成 ✓

EP 统计：
count    261075.000000
mean          0.038723
std           0.035019
min           0.000615
25%           0.015454
50%           0.029236
75%           0.050205
max           0.247919
Name: EP, dtype: float64

BP 统计：
count    261075.000000
mean          0.487918
std           0.322777
min           0.037527
25%           0.260542
50%           0.410610
75%           0.627549
max           2.078216
Name: BP, dtype: float64


In [5]:
# 行业中性化是什么意思？
# 
# 假设你不做中性化，直接用 EP 排序选股：
# 结果可能是：你选出来的"便宜股"全都是银行股，因为银行 PE 普遍低
# 这不是因子的功劳，是行业特性导致的
# 
# 行业中性化的做法：
# 在每个行业内部单独排序，然后再跨行业比较
# 用统计方法：用 EP 对行业虚拟变量做回归，取残差
# 残差 = "去掉行业效应后，这只股票相对同行业的 EP 高低"

def industry_neutralize(group, factor_col):
    """
    对某个月的因子值做行业中性化
    方法：在每个行业内做 z-score 标准化
    （更简单的版本，不需要回归，效果接近）
    
    z-score = (原始值 - 行业均值) / 行业标准差
    结果含义：这只股票的因子值，在它所在行业里偏高还是偏低，偏多少个标准差
    """
    # 计算每个行业的均值和标准差
    industry_mean = group.groupby('industry')[factor_col].transform('mean')
    industry_std = group.groupby('industry')[factor_col].transform('std')
    
    # 避免除以零（某行业只有一只股票时，std=0）
    industry_std = industry_std.replace(0, np.nan)
    
    # 计算行业内 z-score
    neutralized = (group[factor_col] - industry_mean) / industry_std
    return neutralized

print('正在对 EP 做行业中性化...')
df['EP_neutralized'] = df.groupby('trade_date', group_keys=False).apply(
    lambda g: industry_neutralize(g, 'EP')
).values

print('正在对 BP 做行业中性化...')
df['BP_neutralized'] = df.groupby('trade_date', group_keys=False).apply(
    lambda g: industry_neutralize(g, 'BP')
).values

print('行业中性化完成 ✓')

正在对 EP 做行业中性化...
正在对 BP 做行业中性化...
行业中性化完成 ✓


/var/folders/vd/j4l_2tgj3mqb004b4l__22sh0000gn/T/ipykernel_62744/4225265459.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['EP_neutralized'] = df.groupby('trade_date', group_keys=False).apply(
/var/folders/vd/j4l_2tgj3mqb004b4l__22sh0000gn/T/ipykernel_62744/4225265459.py:38: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['BP_neutralized'] = df.groupby('trade_date', group_keys=False).apply(


In [6]:
# 市值中性化是什么意思？
# 
# 小市值股票 EP 和 BP 普遍偏高（因为小公司估值低）
# 如果不做市值中性化，你的因子可能只是在选小市值股票
# 而小市值效应是一个独立的因子，我们不想把它混进来
# 
# 做法：用因子值对 log(市值) 做截面回归，取残差
# 残差 = "去掉市值影响后，这只股票的因子值"

def market_cap_neutralize(group, factor_col):
    """
    用线性回归去掉市值影响
    因变量：因子值
    自变量：log(总市值)
    输出：回归残差（去掉市值影响后的因子值）
    """
    # 删除 NaN，否则回归会报错
    valid = group[[factor_col, 'total_mv']].dropna()
    
    if len(valid) < 10:  # 样本太少就跳过
        return group[factor_col]
    
    # 对市值取对数
    # 为什么取对数：市值分布非常右偏（少数大公司市值极大），
    # 取对数后分布更均匀，回归效果更好
    log_mv = np.log(valid['total_mv'])
    y = valid[factor_col]
    
    # 手动做线性回归（不用 sklearn，只用 numpy）
    # 在 X 矩阵里加一列全为 1 的截距项
    X = np.column_stack([np.ones(len(log_mv)), log_mv])
    
    # OLS 公式：β = (X'X)^(-1) X'y
    # np.linalg.lstsq 是求解最小二乘的函数，自动帮我们算
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    
    # 计算残差 = 实际值 - 预测值
    residuals = y - X @ beta  # @ 是矩阵乘法符号
    
    # 把残差映射回原始索引
    result = group[factor_col].copy()
    result.loc[valid.index] = residuals
    return result

print('正在对 EP 做市值中性化...')
df['EP_final'] = df.groupby('trade_date', group_keys=False).apply(
    lambda g: market_cap_neutralize(g, 'EP_neutralized')
).values

print('正在对 BP 做市值中性化...')
df['BP_final'] = df.groupby('trade_date', group_keys=False).apply(
    lambda g: market_cap_neutralize(g, 'BP_neutralized')
).values

print('市值中性化完成 ✓')

正在对 EP 做市值中性化...
正在对 BP 做市值中性化...
市值中性化完成 ✓


/var/folders/vd/j4l_2tgj3mqb004b4l__22sh0000gn/T/ipykernel_62744/1815434805.py:46: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['EP_final'] = df.groupby('trade_date', group_keys=False).apply(
/var/folders/vd/j4l_2tgj3mqb004b4l__22sh0000gn/T/ipykernel_62744/1815434805.py:51: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['BP_final'] = df.groupby('trade_date', group_keys=False).apply(


In [7]:
# 因子研究的核心逻辑：
# 我们在月末 t 看到因子值，然后观察股票在 t 到 t+1 月之间的收益率
# 看因子值高的股票，收益率是不是真的更高
# 
# 做法：对每只股票，把收益率列"往前移一格"，对齐到上个月的因子值上

# 先按股票和日期排序
df = df.sort_values(['ts_code', 'trade_date']).reset_index(drop=True)

# 计算月度收益率
# pct_change() 计算相邻行之间的变化百分比
# 但我们要在每只股票内部计算，不能跨股票
# groupby('ts_code') 确保只在同一只股票内计算
df['monthly_return'] = df.groupby('ts_code')['close'].pct_change()

# 把收益率往前移一期（shift(-1)）
# 含义：这行记录的是"用上个月的因子值预测这个月的收益"
# 变成：这行记录的是"本月因子值对应的下个月收益"
df['forward_return'] = df.groupby('ts_code')['monthly_return'].shift(-1)

print('未来收益率计算完成 ✓')
print(f'\n月度收益率统计：')
print(df['forward_return'].describe())

未来收益率计算完成 ✓

月度收益率统计：
count    255737.000000
mean          0.009898
std           0.149458
min          -0.830739
25%          -0.066724
50%          -0.003490
75%           0.065537
max          11.056801
Name: forward_return, dtype: float64


In [8]:
# 保存最终的因子数据
# 只保留我们需要的列
cols_to_save = [
    'ts_code', 'trade_date', 'industry', 'total_mv',
    'EP', 'BP',                          # 原始因子值
    'EP_neutralized', 'BP_neutralized',  # 行业中性化后
    'EP_final', 'BP_final',              # 行业+市值中性化后（最终用这个）
    'forward_return'                      # 下个月收益率（预测目标）
]

factor_data = df[cols_to_save].copy()

# 删除 forward_return 为空的行
# 最后一个月的数据没有"下个月收益率"，自然是 NaN，删掉
factor_data = factor_data.dropna(subset=['forward_return'])

factor_data.to_csv('data/factor_data.csv', index=False)

print(f'因子数据已保存到 data/factor_data.csv')
print(f'总行数：{len(factor_data):,}')
print(f'时间范围：{factor_data["trade_date"].min()} 到 {factor_data["trade_date"].max()}')
print(f'\n预览：')
factor_data.head(10)

因子数据已保存到 data/factor_data.csv
总行数：255,737
时间范围：2020-01-23 00:00:00 到 2025-11-28 00:00:00

预览：


,ts_code,trade_date,industry,total_mv,EP,BP,EP_neutralized,BP_neutralized,EP_final,BP_final,forward_return
0,000001.SZ,2020-01-23,银行,3.015680e+07,0.093495,0.905387,-1.178283,-0.865771,-1.194972,-0.887055,-0.066924
1,000001.SZ,2020-02-28,银行,2.813858e+07,0.100200,0.970309,-0.332351,-1.400829,1.209346,0.323847,-0.117241
2,000001.SZ,2020-03-31,银行,2.483958e+07,0.113509,1.099143,0.457825,0.102940,0.763308,0.720817,0.088281
3,000001.SZ,2020-04-30,银行,2.703244e+07,0.108378,1.044714,-0.026166,0.795762,-0.826642,-0.647024,-0.066762
4,000001.SZ,2020-05-29,银行,2.522769e+07,0.116131,1.300898,-0.566198,-0.525567,-1.209937,-1.005554,-0.015385
5,000001.SZ,2020-06-30,银行,2.483958e+07,0.117945,1.119946,1.358663,0.129411,1.381631,-0.653390,0.042187
6,000001.SZ,2020-07-31,银行,2.588749e+07,0.113171,1.074576,0.249655,0.862956,-0.117141,-0.006202,0.130435
7,000001.SZ,2020-08-31,银行,2.926412e+07,0.090452,0.961723,-1.406297,-1.536975,0.630359,-0.052591,0.005968
8,000001.SZ,2020-09-30,银行,2.943878e+07,0.089915,0.956023,-0.445728,0.754121,-0.403047,0.784408,0.170073
9,000001.SZ,2020-10-30,银行,3.444550e+07,0.078304,0.838504,1.084464,1.352451,-1.091805,-0.576926,0.112113
